# Linear Feedback Shift Register

In [ ]:
from itertools import islice

from bits import Bits
from lfsr import primitive_polynomials


In **Linear Feedback Shift Register**, the output of a standard shift register is fed back to its input, so the register can keep evolving cyclically instead of simply shifting data out and stopping. 
The input bit at each clock cycle is formed by combining selected register stages according to a feedback rule. In binary LFSRs, this combination is implemented with XOR, and the selected stages are specified by the feedback polynomial.



1. **Shift-register path**: is the ordinary register mechanism. At each clock cycle, every bit moves by one position along the register chain, and the bit at the output end is shifted out. If no feedback were present, the register contents would eventually disappear after a finite number of shifts.

2. **Linear-feedback path**:  takes selected bits from the current register state, combines them using XOR, and feeds the resulting bit back into the input of the register. This feedback bit becomes the new value shifted into the register at the next clock cycle.
Because XOR is addition modulo 2, this operation is called **linear**. 

Therefore, the **shift-register path provides the state update mechanism**, while the **linear-feedback path determines how the next state depends on the current one**. Together, these two paths generate a **deterministic** sequence of bits that can be very long before repeating.


The **behavior** of the LFSR is **entirely determined** by:

- the **initial register state** (called the seed),
- the **feedback polynomial**.

For a suitable polynomial, an **n-bit LFSR** can cycle through up to **$(2^n - 1)$ nonzero states before repeating**. 

Observation: All-zero state is excluded, because once the register becomes all zeros, the feedback also remains zero and the register can no longer change.

From the block scheme:
$$
\begin{align}
s_j[t] &= s_{j+1}[t-1], \qquad j = 0,1,\ldots,m-2 \\
s_{m-1}[t] &= \bigoplus_{j=0}^{m-1} p_{m-j} \otimes s_j[t-1] \\
b[t] &= s_0[t]\\
\end{align}
$$

![Caption](images/lfsr.png)

**Defining the feedback polynomial**: In this implementation, the feedback polynomial is specified in **sparse form**, that is, as the list of the exponents of its nonzero terms.

For example, $(0, 1, 4)$ represents the polynomial $p(x) = x^4 + x + 1$.

In general, a tuple $(0, a_1, \dots, a_k, \dots, a_N, n)$, where all $a_k$ are distinct integer values such that $0 < a_k < n$, represents the polynomial $p(x) = x^n + \sum_{i=1}^{N} x^{a_i} + 1$


---

**Exercise**

Create a class that implements an LFSR as an iterable.

**Inputs**:
- **Feedback Polynomial**: `set` of `int` representing the degrees of the non-zero coefficients. Example: \{4, 1, 12, 0, 6\} represents $P(x) = x^12 + x^6 + x^4 + x^1 + 1$
- **LFSR state**: the LFSR initial state provided as either a `bytes` string, `int`, or an iterable of `bool`. (optional, default all bits to 1)

**Attributes** (fill free to add any attribute you find useful):
- `poly`: set of the degrees of the non-zero coefficients of the polynomial (`set` of `int`)
- `length`: polynomial degree and length of the shift register (`int`)
- `state`: LFSR state (`Bits`)
- `output`: output bit (`bool`)
- `feedback`: last feedback bit (`bool`)

**Methods** (fill free to add any method you find useful):
- `__init__`: class constructor
- `__len__`: returns the length of the LFSR
- `__iter__`: necessary to be an iterable
- `__next__`: update LFSR state and returns the output bit
- `cycle`: returns a `Bits` object representing the full LFSR cycle starting from the current state, or the state given as an argument (pay attention to the case of non-primitive polynomials)
- `run_steps`: starting from the current state (or the state given as an optional argument) execute N (optional argument, default N=1) LFSR iterations and returns the corresponding output bits as a Bits object.
- `__str__`: return a string describing the LFSR class instance.


### About `@property` and `@output.setter`

In Python, `@property` allows a method behave like an attribute.  
This means you can write `obj.output` instead of `obj.output()`, while still computing the value through a function behind the scenes.

A **setter** defines what happens when that attribute is assigned a value.  
For example, `@output.setter` would handle instructions like `obj.output = ...`.

In this class, `output` is a **read-only derived value**, because it depends on the current state of the register.  
For that reason, the setter does not actually allow assignment: it raises an `AttributeError` to prevent the user from manually changing `output`.

Another example is the initalization of `state`. Since its not an attibute but a propiety, python runs what is specified in `@state.setter`.

In [ ]:
class LFSR:
    '''
    This class implements a binary LFSR. Its internal state is stored in `self.state`,
    while `self.poly` defines the feedback polynomial through its tap positions.
    '''

    def __init__(self, poly, state=None):
        '''
        Initializes the LFSR:
            - `poly` specifies the feedback polynomial in sparse form, that is, as the list of exponents of the nonzero terms.
            - `state` is the initial register state. Default state is all bits set to 1.
            - `self.length` is set to the highest exponent in the polynomial, which determines the register length.
            - `self.poly` and `self.state` are assigned through their setters, so the input is normalized before storage.
        '''

        self.length = max(poly)
        self.poly = poly
        self.state = state

    def __str__(self):
        '''
        Returns a human-readable string showing:
            - the polynomial,
            - the current state,
            - the current output bit.
        '''
        return f'poly: {self.poly}, state: {self.state}, output: {self.output}'

    def __repr__(self):
        '''
            Returns a compact representation of the object: es: LFSR(poly=..., state=...)
        '''
        return f'LFSR(poly={self.poly}, state={self.state})'

    def __len__(self):
        '''
        Returns the register length, that is, the number of bits in the LFSR state.
        '''
        return self.length

    def __iter__(self):
        '''
        Makes the object itself an iterator.
        This allows the LFSR to be used directly in loops or with functions such as next() and islice().
        '''
        return self

    def __next__(self):
        '''
        Advances the LFSR by one clock step. At each call:

            - the feedback bit is computed,
            - that bit is appended to the state,
            - the leftmost bit is removed,
            - the output bit of the current step is returned.
        This method both updates the state and produces one output bit.
        '''

        self.state.append(self.feedback) #Compute and append the feedback bit
        _ = self.state.pop(0) #Remove the leftmost bit
        #es state: [1, 0, 1, 1, 0] -> add feedback bit:[1, 0, 1, 1, 0, 1]  -> pop(0) ->  [0, 1, 1, 0, 1]
        return self.output

    @property
    def state(self):
        '''
        Returns the current register state.
        '''
        return self._state

    @state.setter # function is the write-handler for the property state
    def state(self, state):
        if state is None: # if state is not initialized, set it all to 1s
            state = (1 << len(self)) - 1 #(1 << 5) - 1 = 32 - 1 = 31 -> 0b11111
        if not isinstance(state, Bits): # check is in bit form, otw convert it
            state = Bits(state, length=len(self))
        self._state = state[:len(self)]

    @property
    def poly(self):
        '''
        Returns the feedback polynomial in sparse form.
        '''
        return (self._poly + Bits(1))[::-1].to_sparse()

    @poly.setter
    def poly(self, poly):
        '''
        Converts and stores the polynomial from sparse form in bit form.
        '''
        self._poly = Bits.from_sparse(poly)[1:][::-1]  # remove bit at index 0

    @property # decorator, like this i can just call .feedback istead of .feedback(), even if its a function
    def feedback(self):
        '''
        Computes the feedback bit.
        '''
        # self._state   = 1 0 1 1 0
        # self._poly    = 1 0 0 1 0
        # &             = 1 0 0 1 0
        # XOR           = 0
        return (self._state & self._poly).parity_bit()

    @feedback.setter
    def feedback(self, bit):
        '''
        Raises an error, since the feedback bit is derived from the current state and polynomial and cannot be assigned manually.
        '''
        raise AttributeError("Setting feedback is not allowed")

    @property
    def output(self):
        '''
        Returns the current output bit, defined here as the leftmost bit of the state:
        '''
        return self.state[0]

    @output.setter
    def output(self, bit):
        '''
        Raises an error, since the output bit is derived from the current state and cannot be assigned manually.
        '''
        raise AttributeError("Setting output is not allowed")

    def run_steps(self, N=1):
        '''
        Runs the LFSR for N steps and returns the generated output bits as a Bits object.
        Internally, it repeatedly calls next(self) and collects the returned bits.
        This is useful when a finite output sequence is needed.
        '''
        return Bits([bit for bit in islice(self, N)]) #islcie is an iterable, call method __next__ N times

    def cycle(self, state=None):
        '''
        Generates one full cycle of the LFSR starting from a given state:

            If state is provided, the register is first initialized to that state.
            The initial state is saved.
            Output bits are generated until the state returns to the starting state.
            The collected output sequence is returned.

            This method is useful for studying the period of the LFSR and the sequence produced by a given initialization.
        '''
        if state is not None:
            self.state = state
        state = self.state.copy()
        bits = Bits([next(self)]) #Generate the first output bit (method  generate at least one step before checking )
        while self.state != state:
            bits.append(next(self))
        return bits

### Example $m=3$

- $P(x) = 1 + x + x^3$
- initial state: `111`

|    t |  fb | state | out |
|-----:|-----|-------|-----|
|  `0` | `0` | `111` | `1` |
|  `1` | `1` | `011` | `1` |
|  `2` | `0` | `101` | `1` |
|  `3` | `0` | `010` | `0` |
|  `4` | `1` | `001` | `1` |
|  `5` | `1` | `100` | `0` |
|  `6` | `1` | `110` | `0` |
|  `7` | `0` | `111` | `1` |
|  `8` | `1` | `011` | `1` |
|  `9` | `0` | `101` | `1` |
| `10` | `0` | `010` | `0` |

In [ ]:
# CODE HERE
poly = primitive_polynomials[3][0]
state = None

lfsr = LFSR(poly, state)
print(lfsr)

print(f'  t fb  state    out')
print(f'--------------------')
state = lfsr.state[::-1]
print(f'  0 {lfsr.feedback:2b}  {state} ({state.to_int():x}) {lfsr.output:2b}')
for i, bit in enumerate(islice(lfsr, 10)):
    state = lfsr.state[::-1]
    print(f'{i+1:3d} {lfsr.feedback:2b}  {state} ({state.to_int():x}) {lfsr.output:2b}')

lfsr.cycle()

### Example $m=5$

- $P(x) = 1 + x^2 + x^5$
- initial state: `11111`

|  t |  fb |   state | out | |  t |  fb |   state | out |
|---:|-----|---------|-----|-|---:|-----|---------|-----|
|  0 | `0` | `11111` | `1` | | 16 | `0` | `10100` | `0` |
|  1 | `0` | `01111` | `1` | | 17 | `1` | `01010` | `0` |
|  2 | `1` | `00111` | `1` | | 18 | `1` | `10101` | `1` |
|  3 | `1` | `10011` | `1` | | 19 | `1` | `11010` | `0` |
|  4 | `0` | `11001` | `1` | | 20 | `0` | `11101` | `1` |
|  5 | `1` | `01100` | `0` | | 21 | `1` | `01110` | `0` |
|  6 | `0` | `10110` | `0` | | 22 | `1` | `10111` | `1` |
|  7 | `0` | `01011` | `1` | | 23 | `0` | `11011` | `1` |
|  8 | `1` | `00101` | `1` | | 24 | `0` | `01101` | `1` |
|  9 | `0` | `10010` | `0` | | 25 | `0` | `00110` | `0` |
| 10 | `0` | `01001` | `1` | | 26 | `1` | `00011` | `1` |
| 11 | `0` | `00100` | `0` | | 27 | `1` | `10001` | `1` |
| 12 | `0` | `00010` | `0` | | 28 | `1` | `11000` | `0` |
| 13 | `1` | `00001` | `1` | | 29 | `1` | `11100` | `0` |
| 14 | `0` | `10000` | `0` | | 30 | `1` | `11110` | `0` |
| 15 | `1` | `01000` | `0` | | 31 | `0` | `11111` | `1` |


In [ ]:
# CODE HERE

poly = primitive_polynomials[5][0]
state = None

lfsr = LFSR(poly, state)
print(lfsr)

print(f'  t fb  state      out')
print(f'-----------------------')
state = lfsr.state[::-1]
print(f'  0 {lfsr.feedback:2b}  {state} ({state.to_int():x}) {lfsr.output:3b}')
for i, bit in enumerate(islice(lfsr, 2**max(poly) - 1)):
    state = lfsr.state[::-1]
    print(f'{i+1:3d} {lfsr.feedback:2b}  {state} ({state.to_int():02x}) {lfsr.output:3b}')

lfsr.cycle()

# Alternating Step Generator

An ASG is a **stream cipher keystream generator** composed of three LFSRs:

At each time step:
1. LFSRc is clocked and produces a control bit c.
2. The value of c determines which LSFR is clocked:
   - c = 0 → clock LFSR_0, keep LFSR_1 unchanged
   - c = 1 → clock LFSR_1, keep LFSR_0 unchanged

The output keystream bit is computed as:
$$
b_t = \mathrm{output}(LFSR_0) \oplus \mathrm{output}(LFSR_1)
$$

![Caption](images/ASG.png)

<p style='font-size: 22px'>
    <b> Exercise: </b> 
</p>

Define the class __AlternatingStep__:


### Inputs:
- `seed`: initial value used to set the internal state of the three LFSRs. It can be `None`, a `Bits` object, or any value that can be converted into `Bits`.
- `polyC`: feedback polynomial for the control LFSR. Or a default value
- `poly0`: feedback polynomial for the first data LFSR. Or a default value
- `poly1`: feedback polynomial for the second data LFSR. Or a default value


In [ ]:
## CODE HERE

# Shrinking Generator

The **shrinking generator** consists of two LFSRs:

- **LFSR\_A**, which generates the candidate bits
- **LFSR\_S**, which decides whether each bit produced by **LFSR\_A** is kept or discarded

At each step, both registers are clocked. If the output bit of **LFSR\_S** is `1`, the corresponding bit from **LFSR\_A** is selected and sent to the keystream; if it is `0`, that bit is discarded.

Because this selection process is **irregular**, the output bits are not produced at a constant rate. For this reason, a **FIFO buffer** is needed when a regular output rate is required.

![Caption](images/SG.png)

<p style='font-size: 22px'>
    <b> Exercise: </b> 
</p>

Define the class __ShrinkingGenerator__:

### Inputs

- Feedback polynomials `polyA` and `polyS` (optional, default values are):
  - $ \text{Pa(x)} = x^5 + x^2 + 1 $
  - $ \text{Ps(x)} = x^3 + x + 1 $
- Initial states `stateA` and `stateS` (optional, default: all bits set to 1)

### Attributes

- `lfsrA`: instance of the LFSR class corresponding to $\text{LFSR}_A $
- `lfsrS`: instance of the LFSR class corresponding to $\text{LFSR}_S $
- `output`: boolean storing the last produced output bit



In [ ]:
# CODE HERE

# Trivium

Trivium is a **synchronous stream cipher** based on a **288-bit internal state**. It can be viewed as a single circular shift register or the concatenation of three shift registers (SRa, SRb, and SRc) of different lengths.

Given a **80-bit key** and a **80-bit initialization vector** (IV), Trivium produces a keystream through an **iterative process** which updates the first bit of the three shift register depending on the value of 15 specific bits which for are a feedback (FB) bit, a feedforward (FF) bit, two AND bits and the output bit of each shift register.

| SR | length | feedback | feedforward |      AND |
|:--:|-------:|---------:|------------:|---------:|
|  A |     93 |       69 |          66 |  91,  92 |
|  B |     84 |       78 |          69 |  82,  83 |
|  C |    111 |       87 |          66 | 109, 110 |

The **output** is obtained as a combination of the state bits.

<p >
  <img src="images/Trivium.png" alt="Caption" width="600">
</p>

### Keystream Generation

The keystream generation is described by the following algorithm, in which $z_i$ is the output bit at time instant $i$.

$$
\begin{align*}
& \textbf{for}\; i = 1 \;\textbf{to}\; N \;\textbf{do} \\
&\quad t_1 \leftarrow a_{66} \oplus a_{ 93} \\
&\quad t_2 \leftarrow b_{69} \oplus b_{ 84} \\
&\quad t_3 \leftarrow c_{66} \oplus c_{111} \\
&\quad z_i \leftarrow t_1 \oplus t_2 \oplus t_3 \\
&\quad t_1 \leftarrow t_1 \oplus a_{ 91} \otimes a_{ 92} \oplus b_{78} \\
&\quad t_2 \leftarrow t_2 \oplus b_{ 82} \otimes b_{ 83} \oplus c_{87} \\
&\quad t_3 \leftarrow t_3 \oplus c_{109} \otimes c_{110} \oplus a_{69} \\
&\quad (a_1, a_2, \dots, a_{ 93}) \leftarrow (t_3, a_1, \dots, a_{ 92}) \\
&\quad (b_1, b_2, \dots, b_{ 84}) \leftarrow (t_1, b_1, \dots, b_{ 84}) \\
&\quad (c_1, c_2, \dots, c_{111}) \leftarrow (t_2, c_1, \dots, c_{111}) \\
& \textbf{end for}
\end{align*}
$$

### Initialization

Trivium is inizialiazed by setting:
- the **80-bit key** as the first **80 bits** of SRa and the remaining bits of SRa to `0`.
- the **80-bit IV** as the first **80 bits** of SRb  and the remaining bits of SRb to `0`.
- all bits of SRc to `0` except for the last three bits of SRc which are set to `1`

After setting the initial state, Trivium go through a warm-up of $4\cdot 288 = 1156$ cycles, in which Trivium runs while the output is discarded.

## Homework:

In [ ]:
# CODE HERE